# 05 — Prepare COMPASS figure data

Run this notebook after `01_preprocessing.ipynb`, `02_univariate.ipynb`, and the relevant model-result builds. This is the only Python execution stage for figures. `COMPASS/figures/run_figures.R` consumes its prepared files and never calls Python. No model training or figure rendering occurs here.

Figures cover the ADT cohort (pre-ADT castrate patients included) and the platinum endpoint. Figure 3 cohort sensitivity also prepares platinum and NEPC day-180 incidence summaries across the six ADT definitions, without building six full figure input sets. Federated figures only need the registered federated exports.

The first run reads the ADT longitudinal CSV once, writes a versioned Parquet patient table and `manifest.json`. By default, re-running rebuilds the prepared tables and refreshes the manifest. `COMPASS/figures/run_figures.R` checks this manifest, then prepares/caches its R survival statistics and graphics objects. Missing or changed data require rerunning this notebook. Subsequent rendering reuses those objects.

Caches contain patient data. Keep `FIGURE_DATA_ROOT` under the protected analysis-data tree, not in the figure export directory.

In [ ]:
from pathlib import Path
import os
import sys

# Set before importing Polars. Restart the kernel if Polars was already imported
# and you need to change its thread-pool size.
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", "4"))
os.environ.setdefault("POLARS_MAX_THREADS", str(max(1, min(4, CPUS))))

# Works from this notebook's directory or from the repository root.
HERE = Path.cwd()
SCRIPT_DIR = next((p for p in [HERE.parent / "figures", HERE / "COMPASS" / "figures"]
                   if (p / "prepare_figure_data.py").is_file()), None)
if SCRIPT_DIR is None:
    raise FileNotFoundError("Run from the repository root or COMPASS/notebooks")
sys.path.insert(0, str(SCRIPT_DIR.resolve()))
import importlib
import prepare_figure_data as figure_data
figure_data = importlib.reload(figure_data)
import polars as pl

print(f"Python: {sys.executable}")
print(f"Polars: {pl.__version__}; threads: {pl.thread_pool_size()}")


## Configuration

These defaults match `COMPASS/figures/run_figures.R`. Both honor the `COMPASS_DATA_ROOT`, `COMPASS_FIGURE_DATA_ROOT`, `COMPASS_RHINO_ROOT` and federated path environment variables.

`FORCE` defaults to `True`, so every run rebuilds the prepared tables even when source sizes and timestamps match. Set `COMPASS_PREPARE_OVERWRITE=false` to reuse unchanged tables. This setting is independent of the R image-overwrite switch.

In [ ]:
def env_flag(name, default=False):
    value = os.environ.get(name, str(default)).strip().lower()
    if value not in {"true", "false", "1", "0", "yes", "no"}:
        raise ValueError(f"{name} must be true or false")
    return value in {"true", "1", "yes"}

DATA_ROOT = Path(os.environ.get("COMPASS_DATA_ROOT", "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"))
FIGURE_DATA_ROOT = Path(os.environ.get("COMPASS_FIGURE_DATA_ROOT", str(DATA_ROOT / "figure_data")))
ADT_COHORT_OPTIONS = [
    "adt", "adt_noprecastrate",
    "adt_metastatic_adt", "adt_metastatic_adt_noprecastrate",
    "adt_metastatic_llm", "adt_metastatic_llm_noprecastrate",
]
# Figures are drawn for the ADT cohort and platinum endpoint only (matches run_figures.R).
COHORTS = ["adt"]
ENDPOINTS = ["platinum"]
SCOPE = "all"
FORCE = env_flag("COMPASS_PREPARE_OVERWRITE", True)

CLASSIFIER_PATH = Path("/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_primary_subtype")
PROFILE_DATA_ROOT = Path(os.environ.get("PROFILE_DATA_PATH", "/data/gusev/USERS/jpconnor/data/PROFILE_DATA"))
METASTATIC_SOURCES = {
    "intent": str(DATA_ROOT / "mrn_lists" / "adt_intent_labels_model_cohort.csv"),
    "stage": os.environ.get("COMPASS_REGEX_STAGE_PATH", str(PROFILE_DATA_ROOT / "CANCER_ANNOTATIONS" / "CANCER_STAGE_NOTE_LEVEL.parquet")),
    "llm": os.environ.get("LLM_MET_LABELS_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_met_diagnosis/met_dx_labels.parquet"),
    "icd": str(DATA_ROOT / "prostate_icd_data.csv"),
}
# Federated delivery (DFCI + Fred Hutch + JHU + MSK, rhino_data_v2): the univariate CSV,
# within_site_cox_univariate/, the federated_* model exports, federated_transfer_adt_no_DFCI/
# and figure_data/. For the earlier delivery, point COMPASS_RHINO_ROOT at rhino_results/,
# COMPASS_FEDERATED_RESULTS at its federated_model_results/ CSV and COMPASS_FEDERATED_FIGURE_DATA
# at its misc_figure_data/.
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))
FEDERATED_PATH = Path(os.environ.get("COMPASS_FEDERATED_RESULTS",
    str(RHINO_ROOT / "cox_federated_univariate_adt.csv")))
# 04_federated_transfer.ipynb: bundles refit without Dana-Farber, applied to DFCI endpoints (federated Figure 5)
FEDERATED_TRANSFER_NO_DFCI_PATH = Path(os.environ.get("COMPASS_FEDERATED_TRANSFER_NO_DFCI",
    str(RHINO_ROOT / "federated_transfer_adt_no_DFCI")))
# Aggregate federated KM curves and PSA/testosterone trajectories (federated Figure 3, within-site supplement)
FEDERATED_FIGURE_DATA_PATH = Path(os.environ.get("COMPASS_FEDERATED_FIGURE_DATA",
    str(RHINO_ROOT / "figure_data")))

In [ ]:
config = {
    "data_root": str(DATA_ROOT.resolve()),
    "cache_root": str(FIGURE_DATA_ROOT.resolve()),
    "cohorts": COHORTS,
    "endpoints": ENDPOINTS,
    "scope": SCOPE,
    "force": FORCE,
    "classifier_path": str(CLASSIFIER_PATH),
    "forest_cohorts": ADT_COHORT_OPTIONS,
    "forest_landmark": 180,
    "metastatic": True,
    "metastatic_sources": METASTATIC_SOURCES,
    "federated": True,
    "federated_path": str(FEDERATED_PATH),
    "federated_transfer_no_dfci_path": str(FEDERATED_TRANSFER_NO_DFCI_PATH),
    "federated_figure_data_path": str(FEDERATED_FIGURE_DATA_PATH),
}
CONFIG_PATH = FIGURE_DATA_ROOT / "preparation_config.json"
figure_data.atomic_json(CONFIG_PATH, config)
print(f"Cohorts: {COHORTS}; endpoints: {ENDPOINTS}; scope: {SCOPE}")
print(f"Cache root: {FIGURE_DATA_ROOT}")
print(f"Saved preparation settings: {CONFIG_PATH}")


## Prepare changed data

One scan of the ADT longitudinal CSV prepares:

- one row per patient with treatment/outcome metadata and all-lab record spans;
- metastatic labels (ADT intent, LLM, regex stage) using the longitudinal analysis anchor;
- Figure 3 cohort-sensitivity supplement data: day-180 platinum and NEPC incidence/counts across six ADT definitions, per-patient landmark durations/events for the KM cascade, and Stage-1 ADT-intent/LLM overlap counts. These use the aggregated landmark CSVs and two model-cohort label CSVs in `mrn_lists`; they do not trigger full NEPC or additional cohort renders. The day-180 cohort forest also uses the six existing local univariate Cox result trees, fingerprinted below. `02a_cohort_comparison.ipynb` reports these data as tables only.

Cohort eligibility and the sequencing/Gleason time origins remain defined by the existing analysis inputs. The manifest fingerprints each selected cohort/endpoint result tree and the small cross-cohort/federated inputs used by R.


In [ ]:
import time
started = time.perf_counter()
manifest = figure_data.prepare(config)
print(f"Preparation/validation finished in {time.perf_counter() - started:.2f} seconds")
print(f"Manifest: {FIGURE_DATA_ROOT / 'manifest.json'}")

# Successful arms remain cached if another arm fails. Surface all reported
# errors here so a partial preparation cannot look like a complete run.
if manifest.get("errors"):
    for cell, error in manifest["errors"].items():
        print(f"FAILED {cell}: {error}")
    raise RuntimeError("Some figure inputs failed preparation; see the errors above")


## Check prepared outputs

This summary shows table sizes only, without displaying patient identifiers or measurements. Files remain under the analysis-data cache.


In [ ]:
rows = []
for arm, details in manifest["arms"].items():
    directory = Path(details["directory"])
    for table in figure_data.ARM_TABLES:
        path = directory / f"{table}.parquet"
        n_rows = pl.scan_parquet(path).select(pl.len()).collect().item()
        rows.append({"arm": arm, "table": table, "rows": n_rows,
                     "size_mb": round(path.stat().st_size / 1024**2, 2)})
if manifest.get("cohort_overview"):
    for table in ["incidence"]:
        path = Path(manifest["cohort_overview"]["directory"]) / f"{table}.parquet"
        rows.append({"arm": "Figure 3 cohort sensitivity", "table": table,
                     "rows": pl.scan_parquet(path).select(pl.len()).collect().item(),
                     "size_mb": round(path.stat().st_size / 1024**2, 2)})
if rows:
    display(pl.DataFrame(rows))
else:
    print("Federated-only scope: no patient tables are needed")
print("Prepared result fingerprints:", ", ".join(manifest["cells"]) or "federated comparison only")


## Next: `COMPASS/figures/run_figures.R`

Run `Rscript COMPASS/figures/run_figures.R` with its default `COMPASS_FIGURE_STAGE=all`. The `FIGURES` toggles at the top of that script select figure groups (cohort, LLM, cohort sensitivity, federated univariable/multivariable, transfer survival, transfer checklists); `retained_exports.txt` limits rendered PDFs to the curated ADT set. R checks the manifest/settings and file metadata, prepares and caches the statistical/graphics objects, and renders missing or stale images. It never starts Python. If it reports missing, changed, or incompatible prepared inputs, rerun this notebook.

R writes PDFs into numbered section folders under `<COMPASS_FIG_ROOT>/ADT` (01_cohort, 02_llm, 03_univariate/{cohort_sensitivity,federated}, 04_multivariate/federated, 05_transfer_survival, 06_transfer_checklists), each with a `manuscript/` subfolder for composite figures and a `data/` folder of tables and legends. Federated inputs default to `RHINO_ROOT` (`rhino_data_v2/` under the data root): the across-site univariate CSV, `within_site_cox_univariate/`, the federated Elastic-Net/XGBoost/DeepHit exports, the predictive-checklist exports, aggregate KM/trajectory data in `figure_data/`, and the no-DFCI transfer outputs of `04_federated_transfer.ipynb`. Missing optional inputs are tracked so a later delivery invalidates the cache. Superseded exports are archived recoverably under the data cache, never alongside figures.

After that first R preparation:

- `COMPASS_FIGURE_STAGE=render` re-exports PDFs from the cached snapshot without invoking Python or reading source patient data.
- `COMPASS_FIGURE_STAGE=prepare` refreshes R graphics caches without writing images or preparing Python tables.
- Enabling only federated groups in `FIGURES` skips the DFCI patient-level checks.

After updating analysis inputs, rerun this notebook before the next `run_figures.R` run.